<a href="https://colab.research.google.com/github/seisbench/seisbench/blob/main/examples/03f_active_learning_phasenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Active Learning Loop with SeisBench

This notebook demonstrates SeisBench active learning utilities in `seisbench.generate`.
The example is model-agnostic and focuses on the selection loop, state persistence, and resume workflow.


In [ ]:
# If needed in Colab:
# !pip install seisbench

import numpy as np
import seisbench.generate as sbg

In [ ]:
pool = sbg.ActiveLearningPool(
    dataset_size=30,
    labeled_indices=np.arange(6),
    seed=42,
)


def score_fn(indices: np.ndarray) -> np.ndarray:
    # In practice, compute uncertainty from your model predictions.
    probs = np.tile(np.array([0.7, 0.2, 0.1]), (len(indices), 1))
    for row, idx in enumerate(indices):
        probs[row] = np.roll(probs[row], int(idx) % 3)
    return sbg.UncertaintyQueryStrategy.score_probabilities(probs, method="entropy")


def label_fn(selected_indices: np.ndarray):
    # Replace with your annotation workflow or oracle callback.
    print(f"Label requested for: {selected_indices.tolist()}")


def fit_fn(pool: sbg.ActiveLearningPool, round_result: sbg.RoundResult):
    # Replace with your fine-tuning code.
    print(
        f"Round {round_result.round} | labeled={round_result.labeled_size} | "
        f"unlabeled={round_result.unlabeled_size}"
    )


loop = sbg.ActiveLearningLoop(
    pool=pool,
    score_fn=score_fn,
    label_fn=label_fn,
    fit_fn=fit_fn,
    strategy=sbg.UncertaintyQueryStrategy(method="entropy"),
)

In [ ]:
state_path = "active_learning_state.json"
metrics_path = "active_learning_metrics.csv"

result_1 = loop.run_round(budget=5)
result_2 = loop.run_round(budget=5)

pool.save(state_path)
loop.export_metrics(metrics_path)

result_1, result_2

In [ ]:
# Resume from persisted state
resumed_pool = sbg.ActiveLearningPool.load(state_path)
resumed_loop = sbg.ActiveLearningLoop(
    pool=resumed_pool,
    score_fn=score_fn,
    label_fn=label_fn,
    fit_fn=fit_fn,
    strategy=sbg.UncertaintyQueryStrategy(method="entropy"),
)

resumed_loop.run_round(budget=3)
resumed_loop.export_metrics(metrics_path)